# Assignment 2: IMDB Sentiment Classification
## Deep Neural Network with Comprehensive NLP Preprocessing

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn chardet nltk

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
print("Downloading NLTK resources...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✓ Libraries imported successfully!")

In [ ]:
# Load dataset
# IMPORTANT: Update the path to your CSV file
data = pd.read_csv('IMDB Dataset.csv', encoding='latin-1')

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {data.columns.tolist()}")
print(f"\nSentiment distribution:")
print(data['sentiment'].value_counts())
print(f"\nClass balance: {data['sentiment'].value_counts(normalize=True)}")
print(f"\nFirst 3 reviews:")
display(data.head(3))

In [ ]:
# Check data quality
print("\n" + "="*60)
print("DATA QUALITY CHECK")
print("="*60)

# Missing values
missing_count = data.isnull().sum()
if missing_count.sum() > 0:
    print("\n⚠ Missing Values:")
    print(missing_count[missing_count > 0])
else:
    print("\n✓ No missing values found!")

# Duplicate rows
duplicates = data.duplicated().sum()
print(f"\nDuplicate reviews: {duplicates}")
if duplicates > 0:
    print("Removing duplicates...")
    data = data.drop_duplicates()
    print(f"✓ Removed {duplicates} duplicate rows")
    print(f"New dataset size: {len(data)}")

# Review length statistics
data['review_length'] = data['review'].apply(lambda x: len(x.split()))
print(f"\nReview length statistics (in words):")
print(f"  Mean: {data['review_length'].mean():.1f}")
print(f"  Min: {data['review_length'].min()}")
print(f"  Max: {data['review_length'].max()}")
print(f"  Median: {data['review_length'].median():.1f}")

In [ ]:
# Define comprehensive text preprocessing function
print("\n" + "="*60)
print("NLP PREPROCESSING PIPELINE")
print("="*60)

print("\nPreprocessing steps:")
print("  1. Lowercase conversion")
print("  2. HTML tag removal")
print("  3. URL removal")
print("  4. Special character removal")
print("  5. Number removal")
print("  6. Extra whitespace removal")
print("  7. Tokenization")
print("  8. Stopword removal")
print("  9. Lemmatization")

# Initialize NLP tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Keep some sentiment-bearing words that are usually stopwords
# These are important for sentiment analysis
sentiment_stopwords = {'not', 'no', 'nor', 'neither', 'never', 'none', 'nobody', 'nothing', 
                      'very', 'too', 'more', 'most', 'few', 'less', 'least', 'only', 'just'}
stop_words = stop_words - sentiment_stopwords

print(f"\nStopwords to remove: {len(stop_words)}")
print(f"Sentiment stopwords preserved: {len(sentiment_stopwords)}")

def preprocess_text(text):
    """
    Comprehensive text preprocessing for sentiment analysis
    """
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 4. Remove special characters but keep apostrophes (for contractions)
    text = re.sub(r'[^a-zA-Z\s\'']', '', text)
    
    # 5. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # 6. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 7. Tokenization
    tokens = word_tokenize(text)
    
    # 8. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # 9. Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into text
    cleaned_text = ' '.join(tokens)
    
    return cleaned_text

print("\n✓ Preprocessing function defined")

In [ ]:
# Apply preprocessing
print("\n" + "="*60)
print("APPLYING NLP PREPROCESSING")
print("="*60)

print("\nExample - Before and After Preprocessing:")
print("-" * 60)
sample_review = data['review'].iloc[0]
print(f"ORIGINAL ({len(sample_review.split())} words):")
print(sample_review[:300] + "...")
print("\n" + "-" * 60)

print("\nProcessing all reviews (this may take 1-2 minutes)...")
data['cleaned_review'] = data['review'].apply(preprocess_text)

sample_cleaned = data['cleaned_review'].iloc[0]
print(f"\nCLEANED ({len(sample_cleaned.split())} words):")
print(sample_cleaned[:300] + "...")
print("\n" + "-" * 60)

# Statistics after preprocessing
data['cleaned_length'] = data['cleaned_review'].apply(lambda x: len(x.split()))
print(f"\nPreprocessing Statistics:")
print(f"  Average words before: {data['review_length'].mean():.1f}")
print(f"  Average words after: {data['cleaned_length'].mean():.1f}")
print(f"  Reduction: {((data['review_length'].mean() - data['cleaned_length'].mean()) / data['review_length'].mean() * 100):.1f}%")
print(f"\n✓ Preprocessing complete!")

In [ ]:
# Prepare data
print("\n" + "="*60)
print("DATA PREPARATION")
print("="*60)

# Map sentiment to binary labels
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})

# Use cleaned reviews
reviews = data['cleaned_review'].values
labels = data['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, 
    test_size=0.2, 
    random_state=42, 
    stratify=labels
)

print(f"\nTraining samples: {len(X_train)} ({len(X_train)/len(reviews)*100:.0f}%)")
print(f"Test samples: {len(X_test)} ({len(X_test)/len(reviews)*100:.0f}%)")
print(f"\nClass distribution (Training):")
print(f"  Positive: {np.sum(y_train)} ({np.mean(y_train)*100:.1f}%)")
print(f"  Negative: {len(y_train) - np.sum(y_train)} ({(1-np.mean(y_train))*100:.1f}%)")

In [ ]:
# Tokenize and pad sequences
print("\n" + "="*60)
print("TOKENIZATION AND PADDING")
print("="*60)

max_words = 10000  # Vocabulary size
max_len = 256      # Maximum sequence length

print(f"\nParameters:")
print(f"  Vocabulary size: {max_words}")
print(f"  Max sequence length: {max_len}")

print("\nTokenizing text...")
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print(f"\n✓ Tokenization complete!")
print(f"  Vocabulary size (actual): {len(tokenizer.word_index)}")
print(f"  Training data shape: {X_train_pad.shape}")
print(f"  Test data shape: {X_test_pad.shape}")

# Show tokenization example
print(f"\nExample tokenization:")
sample_text = X_train[0]
sample_tokens = X_train_seq[0]
print(f"  Text: {sample_text[:100]}...")
print(f"  Tokens (first 20): {sample_tokens[:20]}")
print(f"  Padded length: {len(X_train_pad[0])}")

In [ ]:
# Define hyperparameter configurations
print("\n" + "="*60)
print("HYPERPARAMETER EXPERIMENTATION")
print("="*60)

print("\nExperimenting with THREE key hyperparameters:")
print("  1. Activation Function (ReLU, Tanh)")
print("  2. Learning Rate (0.001, 0.01, 0.0001)")
print("  3. Dropout Rate (0.2, 0.3, 0.5)")

configs = [
    # Baseline configurations
    {'name': 'Config 1: ReLU, LR=0.001, Dropout=0.2', 'activation': 'relu', 'learning_rate': 0.001, 'dropout': 0.2},
    {'name': 'Config 2: ReLU, LR=0.001, Dropout=0.3', 'activation': 'relu', 'learning_rate': 0.001, 'dropout': 0.3},
    {'name': 'Config 3: ReLU, LR=0.001, Dropout=0.5', 'activation': 'relu', 'learning_rate': 0.001, 'dropout': 0.5},
    
    # Learning rate variations
    {'name': 'Config 4: ReLU, LR=0.01, Dropout=0.3', 'activation': 'relu', 'learning_rate': 0.01, 'dropout': 0.3},
    {'name': 'Config 5: ReLU, LR=0.0001, Dropout=0.3', 'activation': 'relu', 'learning_rate': 0.0001, 'dropout': 0.3},
    
    # Different activation
    {'name': 'Config 6: Tanh, LR=0.001, Dropout=0.3', 'activation': 'tanh', 'learning_rate': 0.001, 'dropout': 0.3},
    
    # Combined variations
    {'name': 'Config 7: ReLU, LR=0.01, Dropout=0.2', 'activation': 'relu', 'learning_rate': 0.01, 'dropout': 0.2},
    {'name': 'Config 8: ReLU, LR=0.0001, Dropout=0.5', 'activation': 'relu', 'learning_rate': 0.0001, 'dropout': 0.5},
]

print(f"\nTotal configurations to test: {len(configs)}")
print("\nFixed parameters:")
print("  - Embedding dimension: 64")
print("  - Architecture: Embedding → GlobalAvgPooling → Dense[128, 64] → Output")
print("  - Batch size: 32")
print("  - Max epochs: 10 (with early stopping)")

print("\nConfigurations:")
for i, config in enumerate(configs, 1):
    print(f"  {i}. {config['name']}")

In [ ]:
# Train models
print("\n" + "="*60)
print("MODEL TRAINING")
print("="*60)
print("\nNote: Using early stopping (patience=3) to prevent overfitting\n")

results = []

for i, config in enumerate(configs, 1):
    print(f"\n[{i}/{len(configs)}] Training: {config['name']}")
    print("-" * 60)
    
    # Build model
    model = Sequential([
        Embedding(max_words, 64, input_length=max_len),
        GlobalAveragePooling1D(),
        Dense(128, activation=config['activation']),
        Dropout(config['dropout']),
        Dense(64, activation=config['activation']),
        Dropout(config['dropout']),
        Dense(1, activation='sigmoid')
    ])
    
    # Compile with specified learning rate
    optimizer = Adam(learning_rate=config['learning_rate'])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    # Early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    
    # Train model
    history = model.fit(
        X_train_pad, y_train,
        epochs=10,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )
    
    epochs_trained = len(history.history['loss'])
    
    # Predict on test set
    y_pred_prob = model.predict(X_test_pad, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    
    # Store results
    results.append({
        'Config': config['name'],
        'Activation': config['activation'],
        'Learning Rate': config['learning_rate'],
        'Dropout': config['dropout'],
        'Epochs': epochs_trained,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"  Epochs: {epochs_trained:2d} | Acc: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

print("\n" + "="*60)
print("✓ All configurations trained successfully!")
print("="*60)

In [ ]:
# Display results
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS TABLE")
print("="*60)

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values('Accuracy', ascending=False)

print("\n(Sorted by Accuracy - Higher is Better)\n")
display(results_df_sorted[['Config', 'Activation', 'Learning Rate', 'Dropout', 'Accuracy', 'F1 Score', 'ROC-AUC', 'Epochs']])

# Best model
best_config = results_df_sorted.iloc[0]
print("\n" + "="*60)
print("🏆 BEST MODEL")
print("="*60)
print(f"Configuration: {best_config['Config']}")
print(f"\nHyperparameters:")
print(f"  Activation: {best_config['Activation']}")
print(f"  Learning Rate: {best_config['Learning Rate']}")
print(f"  Dropout Rate: {best_config['Dropout']}")
print(f"  Epochs Trained: {int(best_config['Epochs'])}")
print(f"\nPerformance Metrics:")
print(f"  Accuracy: {best_config['Accuracy']:.4f} ({best_config['Accuracy']*100:.2f}%)")
print(f"  Precision: {best_config['Precision']:.4f}")
print(f"  Recall: {best_config['Recall']:.4f}")
print(f"  F1 Score: {best_config['F1 Score']:.4f}")
print(f"  ROC-AUC: {best_config['ROC-AUC']:.4f}")
print("="*60)

In [ ]:
# Analyze hyperparameter impact
print("\n" + "="*60)
print("HYPERPARAMETER IMPACT ANALYSIS")
print("="*60)

# Learning Rate Impact
print("\n1. LEARNING RATE IMPACT:")
print("-" * 60)
lr_analysis = results_df[results_df['Activation'] == 'relu'].groupby('Learning Rate').agg({
    'Accuracy': 'mean',
    'F1 Score': 'mean',
    'ROC-AUC': 'mean'
}).round(4)
print(lr_analysis)
best_lr = lr_analysis['Accuracy'].idxmax()
print(f"\nBest Learning Rate: {best_lr}")

# Dropout Impact
print("\n2. DROPOUT RATE IMPACT:")
print("-" * 60)
dropout_analysis = results_df[results_df['Activation'] == 'relu'].groupby('Dropout').agg({
    'Accuracy': 'mean',
    'F1 Score': 'mean',
    'ROC-AUC': 'mean'
}).round(4)
print(dropout_analysis)
best_dropout = dropout_analysis['Accuracy'].idxmax()
print(f"\nBest Dropout Rate: {best_dropout}")

# Activation Function Impact
print("\n3. ACTIVATION FUNCTION IMPACT:")
print("-" * 60)
activation_analysis = results_df.groupby('Activation').agg({
    'Accuracy': 'mean',
    'F1 Score': 'mean',
    'ROC-AUC': 'mean'
}).round(4)
print(activation_analysis)
best_activation = activation_analysis['Accuracy'].idxmax()
print(f"\nBest Activation: {best_activation}")

In [ ]:
# Visualize results
print("\nGenerating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Learning Rate Impact
lr_data = results_df[results_df['Activation'] == 'relu'].groupby('Learning Rate')['Accuracy'].mean()
axes[0, 0].bar(lr_data.index.astype(str), lr_data.values, color='#FF6B6B', edgecolor='black', alpha=0.8)
axes[0, 0].set_xlabel('Learning Rate', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Average Accuracy', fontweight='bold', fontsize=12)
axes[0, 0].set_title('Learning Rate Impact on Accuracy', fontweight='bold', fontsize=14)
axes[0, 0].set_ylim(0.5, 1.0)
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(lr_data.values):
    axes[0, 0].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# 2. Dropout Rate Impact
dropout_data = results_df[results_df['Activation'] == 'relu'].groupby('Dropout')['Accuracy'].mean()
axes[0, 1].bar(dropout_data.index.astype(str), dropout_data.values, color='#4ECDC4', edgecolor='black', alpha=0.8)
axes[0, 1].set_xlabel('Dropout Rate', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Average Accuracy', fontweight='bold', fontsize=12)
axes[0, 1].set_title('Dropout Rate Impact on Accuracy', fontweight='bold', fontsize=14)
axes[0, 1].set_ylim(0.5, 1.0)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(dropout_data.values):
    axes[0, 1].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# 3. Metrics Comparison (Top 3)
top_3 = results_df_sorted.head(3)
x = np.arange(3)
width = 0.2
axes[1, 0].bar(x - width, top_3['Accuracy'], width, label='Accuracy', color='#45B7D1', edgecolor='black')
axes[1, 0].bar(x, top_3['Precision'], width, label='Precision', color='#95E1D3', edgecolor='black')
axes[1, 0].bar(x + width, top_3['Recall'], width, label='Recall', color='#F38181', edgecolor='black')
axes[1, 0].set_xlabel('Configuration', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('Score', fontweight='bold', fontsize=12)
axes[1, 0].set_title('Top 3 Configurations - Metrics Comparison', fontweight='bold', fontsize=14)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(['Config 1', 'Config 2', 'Config 3'])
axes[1, 0].set_ylim(0.5, 1.0)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. ROC-AUC Comparison
axes[1, 1].barh(range(len(results_df_sorted)), results_df_sorted['ROC-AUC'], color='#FFA07A', edgecolor='black', alpha=0.8)
axes[1, 1].set_yticks(range(len(results_df_sorted)))
axes[1, 1].set_yticklabels([f"Config {i+1}" for i in range(len(results_df_sorted))])
axes[1, 1].set_xlabel('ROC-AUC Score', fontweight='bold', fontsize=12)
axes[1, 1].set_title('ROC-AUC Comparison (All Configs)', fontweight='bold', fontsize=14)
axes[1, 1].set_xlim(0.5, 1.0)
axes[1, 1].grid(axis='x', alpha=0.3)
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig('assignment2_hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Analysis saved as 'assignment2_hyperparameter_analysis.png'")

In [ ]:
# Train final model
print("\n" + "="*60)
print("TRAINING FINAL OPTIMIZED MODEL")
print("="*60)

best_activation = best_config['Activation']
best_lr = best_config['Learning Rate']
best_dropout = best_config['Dropout']

print(f"\nOptimal Hyperparameters:")
print(f"  Activation: {best_activation}")
print(f"  Learning Rate: {best_lr}")
print(f"  Dropout: {best_dropout}")

final_model = Sequential([
    Embedding(max_words, 64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(128, activation=best_activation),
    Dropout(best_dropout),
    Dense(64, activation=best_activation),
    Dropout(best_dropout),
    Dense(1, activation='sigmoid')
])

optimizer = Adam(learning_rate=best_lr)
final_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
history = final_model.fit(
    X_train_pad, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print(f"\n✓ Final model trained successfully!")
print(f"  Epochs: {len(history.history['loss'])}")
print(f"  Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Test with custom reviews
print("\n" + "="*60)
print("TESTING WITH CUSTOM REVIEWS")
print("="*60)

custom_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.",
    "Terrible waste of time. Poor acting, weak storyline, and boring scenes.",
    "An amazing masterpiece! Brilliant direction and outstanding performances.",
    "Disappointing and dull. The movie failed to deliver on its promise."
]

expected_sentiments = ["Positive", "Negative", "Positive", "Negative"]

# Preprocess custom reviews
custom_cleaned = [preprocess_text(review) for review in custom_reviews]
custom_seq = tokenizer.texts_to_sequences(custom_cleaned)
custom_pad = pad_sequences(custom_seq, maxlen=max_len, padding='post')

# Predict
predictions = final_model.predict(custom_pad, verbose=0)

print("\nCustom Review Predictions:")
print("="*60)
for i, (review, pred, expected) in enumerate(zip(custom_reviews, predictions, expected_sentiments), 1):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    confidence = pred[0] if pred[0] > 0.5 else 1 - pred[0]
    match = "✓" if sentiment == expected else "✗"
    
    print(f"\nReview {i}:")
    print(f"Text: {review}")
    print(f"Predicted: {sentiment} (Confidence: {confidence[0]:.2%}) {match}")
    print(f"Expected: {expected}")
    print("-" * 60)

In [ ]:
# Summary
print("\n" + "="*60)
print("ASSIGNMENT 2 COMPLETE ✓")
print("="*60)
print("\nNLP Preprocessing Steps Applied:")
print("  ✓ Lowercase conversion")
print("  ✓ HTML tag removal")
print("  ✓ URL removal")
print("  ✓ Special character removal")
print("  ✓ Number removal")
print("  ✓ Tokenization")
print("  ✓ Stopword removal (with sentiment word preservation)")
print("  ✓ Lemmatization")
print("\nKey Findings:")
print(f"  ✓ Tested {len(configs)} hyperparameter configurations")
print(f"  ✓ Best Learning Rate: {best_config['Learning Rate']}")
print(f"  ✓ Best Dropout Rate: {best_config['Dropout']}")
print(f"  ✓ Best Activation: {best_config['Activation']}")
print(f"  ✓ Best Accuracy: {best_config['Accuracy']:.4f} ({best_config['Accuracy']*100:.2f}%)")
print(f"  ✓ Best ROC-AUC: {best_config['ROC-AUC']:.4f}")
print("\n" + "="*60)